# **Vit Transformer**

Ucitavanje neophodnih biblioteka

In [21]:
import time
import copy
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import timm
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

#### Parametri izvrasavanja programa

In [39]:
MODEL_NAME = "vit_small_patch16_224" #definisme ime modela koji treniramo
DATA_LOCATION = Path("/content/tiny-imagenet-200-modified/tiny-imagenet-200-modified") #lokacija podatak za obuku
OUTPUT_LOCATION = Path("/content/runs/vit") / MODEL_NAME #definisanje imena foldera gde se cuvaju podaci o obucavanju modela
NUM_CLASSES = 200 #broj klasa
IMG_SIZE = 224 #velicina u pikselima
BATCH_SIZE = 256 #velicina jednog batcha
NUM_WORKERS = 8
EPOCHS = 20 #broj epoha
LR = 0.01 #learning rate
WEIGHT_DECAY = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#### Funkcija za ucitavanje podataka

In [36]:
#Funkcija kojom ucitavamo podatke za treniranje i za validaciju
def load_dataset():
    normalization = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) #definisemo parametre za noramlizaciju podataka

    #defomosemo transformacije koje primenjujemo na svaku sliku
    train_transform = transforms.Compose([
        transforms.Resize(224), #resizujemo slike jer mreze ocekuju slike 224x224
        transforms.RandomHorizontalFlip(), #nasumice biramo slike koje cemo horizontalno obrnuti
        #transforms.RandAugment(num_ops=2, magnitude=15), #primenjujemo nasumicne transformacije, parametri iz papira
        transforms.ToTensor(),
        normalization,
        #transforms.RandomErasing(p=0.25) #brismeo 25% nasumicno izabranog dela slike
    ])

    #kao prethodna funkcija samo za validaciju
    validation_transorm = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        normalization
    ])

    #ImageFolder ocekuje posbnu strukturu foldera koji sadrzi podatke
    train_dataset = datasets.ImageFolder(DATA_LOCATION / "train", transform = train_transform)
    validation_dataset = datasets.ImageFolder(DATA_LOCATION / "val", transform = validation_transorm)

    #definismo interabilnu strkturu koja sadrzi podatke organizovane u batchove, za treniranje i za validaciju
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

    return train_loader, validation_loader


#### Funkcija za obucavanje modela

In [28]:
#Funkcija za obucavanje modela (sa mixed precision / AMP)
def training_run(model, loader, lossFunction, optimizer, mixcut):

    model.train()

    #inicijalizacija promenjivih.
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

        #images, labels = mixcut(images, labels)

        #brisemo akomuliran gradijent
        optimizer.zero_grad()

        #forward pass u BF16 precision umesto FP32 - brze na A100 tensor core-ovima
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            model_predictons = model(images)  # model pravi predvidjanje
            loss = lossFunction(model_predictons, labels)  # racunamo gresku


        loss.backward()   # racunamo gradijent
        optimizer.step() # optimizacioni korak

        total_loss += loss.item() * images.size(0)  # ukupna greska za ceo batch #.argmax(dim=1) obrisano jer se ne korsiti mixup
        correct += (model_predictons.argmax(dim=1) == labels).sum().item()  # broj ispravnih predvidjanja u batchu
        total += images.size(0)  # ukupno slika u batchu

    train_loss = total_loss / total  # prosecna greska za epohu
    train_accuracy = correct / total  # ukupno

    return train_loss, train_accuracy

#### Funkcija za evaluaciju modela

In [ ]:
#Funkcija za evaluaciju modela
def validation_run(model, loader, lossFunction):

    #evaluiramo model
    model.eval()

    #incijalizacija parametara
    total_loss, correct, total = 0.0, 0, 0

    #prolazimo validacioni skup
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            #predvidjanje modela
            model_predictons = model(images)

            #greska predvidjanja
            loss = lossFunction(model_predictons,labels)

            #ukupan gubitak na jednom batchu
            total_loss += loss.item() * images.size(0)
            #korektna predvidjanja na celom batchy
            correct += (model_predictons.argmax(dim=1) == labels).sum().item()
            #ukupno sika u batchu
            total += images.size(0)

    #prosecna greska na batchu
    validation_loss = total_loss/total
    #validaciona greska na batcu
    validation_accuracy = correct/total

    return validation_loss, validation_accuracy

#### Funkcija gde se vrsi celokupno obucavanje modela

In [40]:
#Funkcija u kojoj se vrsi obucavanje modela
def train():

    #dictionary koji sadrzi metrike
    metrics = {
    "epoch": [],
    "training_loss": [],
    "training_accuracy": [],
    "validation_loss": [],
    "validation_accuracy": [],
    "current_lr": [],
    "epoch_time": [] #vreme potrebno za jednu epohu
    }

    #ucitavamo podatke
    train_loader, validation_loader = load_dataset()

    #stampamo podatke na izlaz
    print(f"Model name: {MODEL_NAME}")
    print(f"Device: {DEVICE}")
    print(f"Training samples: {len(train_loader.dataset)}")
    print(f"Validation samples: {len(validation_loader.dataset)}")

    #definisemo model. Korisitmo timm bibilotek. Pretrained je false s obzirom da obucavamo model od 0
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_path_rate=0.2)
    model.to(DEVICE)

    for paramaters in model.parameters():
        paramaters.requires_grad = False

    for head in model.get_classifier().parameters():
        head.requires_grad = True

    #stampamo broj parametara
    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable model params: {trainable_parameters/1e6:.1f}M")

    mixup = v2.MixUp(alpha=0.8, num_classes=NUM_CLASSES)
    cutmix = v2.CutMix(alpha=1.0, num_classes=NUM_CLASSES)

    mixcut = v2.RandomChoice([mixup, cutmix])

    #definisemo algoritam za obucavanje, prosledjujemo samo head
    optimizer = torch.optim.AdamW( model.get_classifier().parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    #scheduler za adaptiranje learning rate
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    #definisemo funkciju greske
    lossFunction = nn.CrossEntropyLoss(label_smoothing=0.1)

    #inicijalizacija promenljivih
    best_validation_accuracy = 0.0
    best_model_state = None

    #prolazimo kroz epohe
    for epoch in range(EPOCHS):

        #pocinjemo da merimo vreme
        start_time = time.time()

        #obucavanje modela, validacija modela, i updajtovanje scheduler-a
        training_loss, training_accuracy = training_run(model, train_loader, lossFunction, optimizer,mixcut)
        validation_loss, validation_accuracy = validation_run(model, validation_loader, lossFunction)
        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

        #prestajemo da merimo vreme i racunamo koliko je bilo potrebno
        epoch_train_time = time.time() - start_time

        #updejtujemo metrike
        metrics["epoch"].append(epoch + 1)
        metrics["training_loss"].append(training_loss)
        metrics["training_accuracy"].append(training_accuracy)
        metrics["validation_loss"].append(validation_loss)
        metrics["validation_accuracy"].append(validation_accuracy)
        metrics["current_lr"].append(current_lr)
        metrics["epoch_time"].append(epoch_train_time)

        #stampamo na izlaz podatke jede epohe
        print(f"epoch {epoch + 1}/{EPOCHS} train_loss={training_loss:.4f} train_acc={training_accuracy:.4f} "
                      f"val_loss={validation_loss:.4f} val_acc={validation_accuracy:.4f} time={epoch_train_time:.1f}s")

        #ako je rezultat na validacionom skup bio bolji od prethodnog rezultata, taj model cuvamo
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
            best_model_state = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 10 == 0:
            checkpoint_path = OUTPUT_LOCATION / f"checkpoint_epoch_{epoch + 1}.pth"
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "validation_accuracy": validation_accuracy,
            }, checkpoint_path)
            print(f"Checkpoint saved: {checkpoint_path}")

    return best_model_state, best_validation_accuracy, metrics


#### Funkcije za cuvanje modela i statistike obucavanja

In [31]:
#funkcija za cuvanje modela
def save_model(best_model_state):
    best_model_path = OUTPUT_LOCATION / "best_model.pth"
    torch.save(best_model_state, best_model_path)
    return

#funkcija za generisanje statistike obucavanja modela
def generate_statistics(metrics):
    history_path = OUTPUT_LOCATION / "metrics.csv"
    df = pd.DataFrame(metrics)
    df.to_csv(history_path, index=False)
    return


#### Ucitavanje podataka

In [ ]:
from google.colab import files
uploaded = files.upload()  # opens a file picker, select subset.zip

Saving tiny-imagenet-200-modified.zip to tiny-imagenet-200-modified.zip


In [ ]:
!unzip -q tiny-imagenet-200-modified.zip -d /content/tiny-imagenet-200-modified

#### Pokretanje programa

In [41]:
OUTPUT_LOCATION.mkdir(parents=True, exist_ok=True) #kreiramo dirketorujum
best_model_state, best_validation_accuracy, metrics = train() #obucavamo model
print(f"\nModel done with training.")
save_model(best_model_state) #cuvamo model
print(f"\nBest model saved.")
generate_statistics(metrics) #generisemo statistiku
print(f"\nTraining metrics saved.")

Model name: vit_small_patch16_224
Device: cuda
Training samples: 95000
Validation samples: 10000
Trainable model params: 0.1M
epoch 1/20 train_loss=4.6237 train_acc=0.2879 val_loss=3.4603 val_acc=0.6253 time=44.2s
epoch 2/20 train_loss=4.6530 train_acc=0.3158 val_loss=3.4261 val_acc=0.6366 time=45.6s
epoch 3/20 train_loss=4.6424 train_acc=0.3187 val_loss=3.4912 val_acc=0.6335 time=45.2s
epoch 4/20 train_loss=4.5773 train_acc=0.3220 val_loss=3.3985 val_acc=0.6400 time=45.2s
epoch 5/20 train_loss=4.5090 train_acc=0.3243 val_loss=3.3712 val_acc=0.6302 time=45.1s
epoch 6/20 train_loss=4.4097 train_acc=0.3273 val_loss=3.2729 val_acc=0.6360 time=45.5s
epoch 7/20 train_loss=4.2475 train_acc=0.3344 val_loss=3.1001 val_acc=0.6478 time=45.2s
epoch 8/20 train_loss=4.1332 train_acc=0.3391 val_loss=3.0365 val_acc=0.6535 time=44.4s
epoch 9/20 train_loss=3.9890 train_acc=0.3513 val_loss=2.9158 val_acc=0.6624 time=45.5s
epoch 10/20 train_loss=3.8487 train_acc=0.3588 val_loss=2.8217 val_acc=0.6633 time

In [42]:
!zip -r /content/vit_pre.zip /content/runs/vit

  adding: content/runs/vit/ (stored 0%)
  adding: content/runs/vit/vit_small_patch16_224/ (stored 0%)
  adding: content/runs/vit/vit_small_patch16_224/best_model.pth (deflated 7%)
  adding: content/runs/vit/vit_small_patch16_224/checkpoint_epoch_20.pth (deflated 7%)
  adding: content/runs/vit/vit_small_patch16_224/metrics.csv (deflated 49%)
  adding: content/runs/vit/vit_small_patch16_224/checkpoint_epoch_10.pth (deflated 7%)


In [43]:
from google.colab import files
files.download('/content/vit_pre.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>